# SFT 概念与原理

监督微调（Supervised Fine-Tuning，SFT）沿用预训练阶段的 next-token 预测目标，驱动模型适配目标数据集所期望的行为分布。为直观呈现其技术内核，本节以同一条 Wordle 对话为线索，系统剖析 **偏移标签（Shifted Labels）、交叉熵损失（Cross-Entropy）与标签掩码（Label Masking）** 的协同配合机制。

## 1. 例子

```text
预训练（Pretraining）：
  输入："The capital of France is"
  输出："Paris."
  学的是：语言统计规律（什么词通常接在什么词后面）

监督微调（SFT）：
  输入：[user] "Round 1 feedback: X-X-X-X-X. Guess again."
  输出：[assistant] "<think>...</think><guess>[slime]</guess>"
  学的是：课程数据中的对话行为模式和输出协议
```

## 2. 共同的 next-token Cross-Entropy

预训练与本课程采用的 SFT 共享同一自回归 next-token 预测框架，优化目标均为交叉熵损失。具体地，给定长度为 \(n+1\) 的 token 序列 \(t_0, t_1, ..., t_n\)，模型以前缀 \(t_0 ... t_{n-1}\) 作为输入，对应标签则右移一位设为 \(t_1 ... t_n\)，驱动模型在每个位置预测其后继词元。

对话数据涉及 system、user 与 assistant 三类角色。本课程的监督策略为**仅对 assistant 回复计算损失**：实现上，将 system 与 user 对应位置的标签统一置为 ignore，PyTorch 交叉熵接口在计算时会自动跳过这些掩码位置；仅保留 assistant 部分的 next-token 标签用于梯度回传与参数更新。

![SFT label masking：system/user 位置忽略 loss，assistant 位置计算 loss](./images/sft-label-masking.png)

图中每个方框代表一段完整消息，而非单个 token。system/user 内容仍作为上下文参与模型前向计算、影响 assistant 的预测分布；掩码仅决定相应目标位置是否计入损失，不影响模型对上下文的感知。

---


## 4. 如何评估 SFT 效果？

评价 Wordle 游戏成功的核心量化指标是答案正确率衡量模型在单局游戏中的最终结果：每局游戏最多进行 6 轮猜测，若模型在 6 轮内猜中秘密词，则该局 correct_answer = 1；否则为 0。但是为了得到正确答案，需要2个阶段。

### 格式分（`format_reward`）

`format_reward` 用于衡量模型回复是否符合预期的 XML 格式（如是否包含可解析的 `<guess>` 字段）。它是一个 0~1 之间的分级得分，由解析器根据回复能否被成功解析给出。该指标仅反映输出的**格式合规性**，不涉及猜词内容是否正确。在 SFT 阶段，我们主要关注模型能否生成环境可读的回复，因此格式分是首要的量化指标。

### 猜中率（`correct_answer`）

`correct_answer` 用于衡量模型在 Wordle 游戏中的真实表现。每局游戏中，模型在 6 轮内猜中秘密词则得 1 分，否则得 0 分。在固定评测集上对所有局取平均，即得到 **6 轮猜中率**。该指标直接反映猜词策略的优劣。由于涉及策略与能力提升，不是 SFT 阶段阶段

---

综上，SFT 阶段只需确保模型具备良好的格式遵从能力（高格式分），而猜词策略的提升则留待后续 RL 阶段完成。。
---

## 小结

- 预训练与本课程 SFT 共享 shifted next-token CE 目标，差异主要来自数据构成与监督区间。
- `-100` 掩码使非 assistant 位置不直接贡献损失，但相关 token 仍参与模型输入。
- SFT 能够模仿标签中的回复协议与行为模式；规则理解与策略表现须单独评测。
- 下一章将介绍本课程训练 SFT 所使用的框架——TorchTitan。

## 练习

1. （单选题）本课程 Wordle SFT 与典型 causal LM 预训练都使用 next-token CE；本课程 recipe 的关键区别是？
    A. Pretraining 使用 MSE Loss，SFT 使用 CE Loss
    B. 它使用多轮对话，并把非 assistant 目标位置设为 -100
    C. Pretraining 不需要训练数据，SFT 需要
    D. 两者完全相同

2. （单选题）user 消息对应的 label 设为 `-100` 后，这些 token 在训练中的作用是什么？
    A. 仍作为上下文，相关目标位置不计入 CE loss
    B. 从模型输入中删除，只保留 assistant token
    C. 作为额外奖励参与策略优化
    D. 只用于计算 user 消息的 CE loss

3. （多选题）仅根据 SFT 训练目标，可以合理预期什么？
    A. 模仿 `<think>...</think><guess>[word]</guess>` 等标签协议
    B. 模仿训练轨迹中反馈与下一次猜测的关联
    C. 无须评测即可证明模型真正理解 Wordle 规则
    D. 保证学到 6 轮内猜对的最优策略


In [ ]:
!cat ./answer/01.03_answer.txt